# Compute contribution decay, inspect it, then write to CockroachDB

This notebook deliberately separates algorithm computation from database writes:

1. Compute decay results and save bounded Parquet parts.
2. Inspect and manually verify those results.
3. Run a separate explicit cell to replace and write the verified results to CockroachDB.


In [21]:
# Configuration
from datetime import datetime, timezone
from pathlib import Path

SCOPE = "project"  # "project" or "global"
PROJECT_KEYWORD = "D3lMundos"  # only used for project scope
SOURCE_BATCH_SIZE = 100_000
WRITE_BATCH_SIZE = 100_000
INCLUDE_ACTIVE_MULTIPLIERS = False

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = Path("output") / f"{SCOPE}_{PROJECT_KEYWORD}_{RUN_ID}"

assert SCOPE in {"project", "global"}
if SCOPE == "project":
    assert PROJECT_KEYWORD

print("Computed results will be stored in:", OUTPUT_DIR)


Computed results will be stored in: output/project_D3lMundos_20260615T045541Z


In [22]:
# Imports
from importlib import reload
from inspect import getsource
from time import perf_counter

import polars as pl
import mindshare_compute.db as db_module

# Reload because Jupyter caches imported modules between runs.
reload(db_module)

from mindshare_compute.decay import DecayComputer

iter_decay_source = db_module.iter_decay_source
DecayResultWriter = db_module.DecayResultWriter

pl.Config.set_tbl_rows(20)
assert "iter_slices" in getsource(DecayResultWriter.write_batch)
assert not hasattr(DecayResultWriter, "_write_batch_copy")
print("Loaded bounded multi-row INSERT writer from:", db_module.__file__)


Loaded bounded multi-row INSERT writer from: /home/babin411/Nucleus/mindshare-backend-optimization/mindshare_compute/db.py


## 1. Compute algorithm results only

This cell reads source rows and computes the decay algorithm. It **does not write to CockroachDB**. Each computed batch is saved as a Parquet part so the complete result can be inspected without holding it all in WSL memory.


In [23]:
computer = DecayComputer(
    SCOPE,
    include_active_multipliers=INCLUDE_ACTIVE_MULTIPLIERS,
)
source = iter_decay_source(
    SCOPE,
    PROJECT_KEYWORD if SCOPE == "project" else None,
    target="crdb",
    batch_size=SOURCE_BATCH_SIZE,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
compute_start = perf_counter()
compute_seconds = 0.0
rows_computed = 0
parts_written = 0
last_computed_batch = None

for batch_number, source_batch in enumerate(source):
    started = perf_counter()
    result_batch = computer.process_batch(source_batch)
    compute_seconds += perf_counter() - started

    result_batch.write_parquet(OUTPUT_DIR / f"part-{batch_number:05d}.parquet")
    rows_computed += result_batch.height
    parts_written += 1
    last_computed_batch = result_batch
    print(f"computed part={batch_number:05d} total_rows={rows_computed:,}")

compute_summary = pl.DataFrame({
    "scope": [SCOPE],
    "project_keyword": [PROJECT_KEYWORD if SCOPE == "project" else None],
    "rows_computed": [rows_computed],
    "parquet_parts": [parts_written],
    "algorithm_compute_seconds": [compute_seconds],
    "compute_and_parquet_wall_seconds": [perf_counter() - compute_start],
})
compute_summary


computed part=00000 total_rows=40,902


scope,project_keyword,rows_computed,parquet_parts,algorithm_compute_seconds,compute_and_parquet_wall_seconds
str,str,i64,i64,f64,f64
"""project""","""D3lMundos""",40902,1,0.449141,59.700903


## 2. Manually inspect and verify

These cells scan the saved Parquet files lazily. Nothing is written to CockroachDB here. Add any manual checks you need before running the database-write section.


In [24]:
result_scan = pl.scan_parquet(OUTPUT_DIR / "*.parquet")

result_scan.head(50).collect()


project_keyword,reply_post_id,original_post_id,replier_x_id,original_author_x_id,post_created_at,replier_base_score,effective_score,contribution_score,reply_number,local_reply_count,decay_type
str,str,str,str,str,"datetime[μs, UTC]",f64,f64,f64,i64,i64,str
"""D3lMundos""","""2023499772662010185""","""2023498292072755352""","""1000030347516497921""","""777374198108872705""",2026-02-16 20:48:35 UTC,163.24,163.24,163.24,1,1,"""FIRST_REPLY"""
"""D3lMundos""","""1981240931580821639""","""1981151194878980114""","""1000110853470019584""","""1974840743790325761""",2025-10-23 06:07:02 UTC,0.0,0.0,0.0,1,1,"""FIRST_REPLY"""
"""D3lMundos""","""2018361856033710343""","""2018360300509274311""","""1000110853470019584""","""1985421410601484290""",2026-02-02 16:32:20 UTC,0.0,0.0,0.0,2,1,"""FIRST_REPLY"""
"""D3lMundos""","""2018364715181769054""","""2018360300509274311""","""1000110853470019584""","""1985421410601484290""",2026-02-02 16:43:42 UTC,0.0,0.0,0.0,3,2,"""LOCAL_DECAY"""
"""D3lMundos""","""2020472776616796587""","""2020454686101217721""","""1000110853470019584""","""1519765069663551489""",2026-02-08 12:20:23 UTC,0.0,0.0,0.0,4,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""2021614771519311916""","""2021614466098426147""","""1000110853470019584""","""1992969366368120832""",2026-02-11 15:58:15 UTC,0.0,0.0,0.0,5,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""2022353739873394830""","""2022347522321301596""","""1000110853470019584""","""2018582582682963968""",2026-02-13 16:54:39 UTC,0.0,0.0,0.0,6,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""2023711818917593220""","""2023701806522790309""","""1000110853470019584""","""1455968407937900545""",2026-02-17 10:51:11 UTC,0.0,0.0,0.0,7,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""1982983460277952781""","""1982824934155141343""","""1000466366451933185""","""1932741925611925504""",2025-10-28 01:31:13 UTC,65.63,65.63,65.63,1,1,"""FIRST_REPLY"""


In [25]:
# Useful high-level verification before writing.
result_scan.group_by("decay_type").agg(
    pl.len().alias("rows"),
    pl.col("contribution_score").min().alias("min_score"),
    pl.col("contribution_score").mean().alias("mean_score"),
    pl.col("contribution_score").max().alias("max_score"),
).sort("decay_type").collect()


decay_type,rows,min_score,mean_score,max_score
str,u32,f64,f64,f64
"""FIRST_REPLY""",10717,0.0,202.609175,3184.36
"""GLOBAL_DECAY""",19623,0.0,112.168692,2865.92
"""LOCAL_DECAY""",10562,0.0,33.07047,1163.58


In [26]:
# Change these filters for a replier or author you want to verify manually.
VERIFY_REPLIER_X_ID = None
VERIFY_ORIGINAL_AUTHOR_X_ID = None

verification = result_scan
if VERIFY_REPLIER_X_ID:
    verification = verification.filter(pl.col("replier_x_id") == VERIFY_REPLIER_X_ID)
if VERIFY_ORIGINAL_AUTHOR_X_ID:
    verification = verification.filter(
        pl.col("original_author_x_id") == VERIFY_ORIGINAL_AUTHOR_X_ID
    )

verification.sort("replier_x_id", "post_created_at").head(200).collect()


project_keyword,reply_post_id,original_post_id,replier_x_id,original_author_x_id,post_created_at,replier_base_score,effective_score,contribution_score,reply_number,local_reply_count,decay_type
str,str,str,str,str,"datetime[μs, UTC]",f64,f64,f64,i64,i64,str
"""D3lMundos""","""2023499772662010185""","""2023498292072755352""","""1000030347516497921""","""777374198108872705""",2026-02-16 20:48:35 UTC,163.24,163.24,163.24,1,1,"""FIRST_REPLY"""
"""D3lMundos""","""1981240931580821639""","""1981151194878980114""","""1000110853470019584""","""1974840743790325761""",2025-10-23 06:07:02 UTC,0.0,0.0,0.0,1,1,"""FIRST_REPLY"""
"""D3lMundos""","""2018361856033710343""","""2018360300509274311""","""1000110853470019584""","""1985421410601484290""",2026-02-02 16:32:20 UTC,0.0,0.0,0.0,2,1,"""FIRST_REPLY"""
"""D3lMundos""","""2018364715181769054""","""2018360300509274311""","""1000110853470019584""","""1985421410601484290""",2026-02-02 16:43:42 UTC,0.0,0.0,0.0,3,2,"""LOCAL_DECAY"""
"""D3lMundos""","""2020472776616796587""","""2020454686101217721""","""1000110853470019584""","""1519765069663551489""",2026-02-08 12:20:23 UTC,0.0,0.0,0.0,4,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""2021614771519311916""","""2021614466098426147""","""1000110853470019584""","""1992969366368120832""",2026-02-11 15:58:15 UTC,0.0,0.0,0.0,5,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""2022353739873394830""","""2022347522321301596""","""1000110853470019584""","""2018582582682963968""",2026-02-13 16:54:39 UTC,0.0,0.0,0.0,6,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""2023711818917593220""","""2023701806522790309""","""1000110853470019584""","""1455968407937900545""",2026-02-17 10:51:11 UTC,0.0,0.0,0.0,7,1,"""GLOBAL_DECAY"""
"""D3lMundos""","""1982983460277952781""","""1982824934155141343""","""1000466366451933185""","""1932741925611925504""",2025-10-28 01:31:13 UTC,65.63,65.63,65.63,1,1,"""FIRST_REPLY"""


## 3. Write verified Parquet results to CockroachDB

Run this cell only after manually verifying the algorithm output.

- Project scope deletes that project's existing rows from `mindshare_score_test.test_contribution_scores` before writing.
- Global scope truncates `mindshare_score_test.test_global_contribution_scores` before writing.
- The writer creates the schema/table when they do not exist.


In [27]:
# Explicit safety switch: change to True only after inspecting the results above.
WRITE_VERIFIED_RESULTS_TO_CRDB = True

if not WRITE_VERIFIED_RESULTS_TO_CRDB:
    raise RuntimeError(
        "Set WRITE_VERIFIED_RESULTS_TO_CRDB = True after manually verifying the Parquet results."
    )

write_start = perf_counter()
write_seconds = 0.0
rows_written = 0

# collect_batches streams the verified Parquet dataset without loading it all.
parquet_batches = pl.scan_parquet(OUTPUT_DIR / "*.parquet").collect_batches(
    chunk_size=WRITE_BATCH_SIZE,
    maintain_order=True,
)

with DecayResultWriter(
    SCOPE,
    PROJECT_KEYWORD if SCOPE == "project" else None,
    use_copy=False,
) as writer:
    for batch_number, result_batch in enumerate(parquet_batches):
        started = perf_counter()
        rows_written += writer.write_batch(result_batch)
        write_seconds += perf_counter() - started
        print(f"wrote batch={batch_number:05d} total_rows={rows_written:,}")

write_summary = pl.DataFrame({
    "rows_written": [rows_written],
    "cockroach_write_seconds": [write_seconds],
    "write_wall_seconds": [perf_counter() - write_start],
})
write_summary


wrote batch=00000 total_rows=40,902


rows_written,cockroach_write_seconds,write_wall_seconds
i64,f64,f64
40902,40.88677,66.353668
